# Manufacturing Quality Control Dashboard
Generates an interactive HTML dashboard from Gold layer tables and saves it to the lakehouse.

In [ ]:
# Read all Gold tables
df_production = spark.read.format('delta').table('gold_production_daily_summary').toPandas()
df_equipment = spark.read.format('delta').table('gold_equipment_health_daily').toPandas()
df_shift = spark.read.format('delta').table('gold_shift_performance').toPandas()
df_product = spark.read.format('delta').table('gold_product_quality').toPandas()
df_weekly = spark.read.format('delta').table('gold_weekly_trends').toPandas()
df_scorecard = spark.read.format('delta').table('gold_line_scorecard').toPandas()
print('All Gold tables loaded')

In [ ]:
import json
from datetime import datetime

# Calculate KPI values
total_units = int(df_scorecard['total_units_all_time'].sum())
total_defects = int(df_scorecard['total_defects_all_time'].sum())
avg_oee = round(df_scorecard['overall_oee'].mean(), 1)
avg_yield = round(df_scorecard['overall_yield'].mean(), 1)
total_lines = len(df_scorecard)
best_line = df_scorecard.loc[df_scorecard['overall_oee'].idxmax()]
worst_line = df_scorecard.loc[df_scorecard['overall_oee'].idxmin()]

# Daily OEE trend by line
oee_by_date = df_production.groupby(['production_date', 'production_line'])['oee'].mean().reset_index()
oee_lines = {}
for line in oee_by_date['production_line'].unique():
    line_data = oee_by_date[oee_by_date['production_line'] == line].sort_values('production_date')
    oee_lines[line] = {'dates': line_data['production_date'].tolist(), 'values': [round(v, 1) for v in line_data['oee'].tolist()]}

# Equipment health distribution
health_dist = df_equipment.groupby('health_status')['machine_id'].nunique().to_dict()
health_colors = {'Healthy': '#22c55e', 'Warning': '#f59e0b', 'At Risk': '#f97316', 'Critical': '#ef4444'}

# Shift comparison
shift_data = df_shift.groupby('shift').agg({'total_units': 'sum', 'total_defects': 'sum', 'avg_yield': 'mean'}).reset_index()

# Product quality
product_data = df_product.sort_values('quality_rank')[['product', 'avg_yield', 'total_units', 'quality_tier', 'quality_rank']].to_dict('records')

# Weekly trends
weekly_agg = df_weekly.groupby('production_week').agg({'weekly_avg_oee': 'mean', 'weekly_units': 'sum'}).reset_index().sort_values('production_week')

# OEE class distribution
oee_class_dist = df_production.groupby('oee_class').size().to_dict()
oee_class_colors = {'World Class': '#22c55e', 'Acceptable': '#3b82f6', 'Needs Improvement': '#f59e0b', 'Critical': '#ef4444'}

print(f'KPIs calculated: OEE={avg_oee}%, Yield={avg_yield}%, Units={total_units:,}')

In [ ]:
html = f'''
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Manufacturing Quality Control Dashboard</title>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js"></script>
<style>
  * {{ margin: 0; padding: 0; box-sizing: border-box; }}
  body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; background: #f1f5f9; color: #1e293b; }}
  .header {{ background: linear-gradient(135deg, #1e3a5f 0%, #0f766e 100%); color: white; padding: 24px 32px; }}
  .header h1 {{ font-size: 24px; font-weight: 700; }}
  .header p {{ font-size: 14px; opacity: 0.8; margin-top: 4px; }}
  .container {{ max-width: 1400px; margin: 0 auto; padding: 24px; }}
  .kpi-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 16px; margin-bottom: 24px; }}
  .kpi-card {{ background: white; border-radius: 12px; padding: 20px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); }}
  .kpi-card .label {{ font-size: 12px; color: #64748b; text-transform: uppercase; letter-spacing: 0.05em; font-weight: 600; }}
  .kpi-card .value {{ font-size: 32px; font-weight: 700; margin: 8px 0 4px; }}
  .kpi-card .sub {{ font-size: 13px; color: #94a3b8; }}
  .kpi-card.green .value {{ color: #16a34a; }}
  .kpi-card.blue .value {{ color: #2563eb; }}
  .kpi-card.amber .value {{ color: #d97706; }}
  .kpi-card.red .value {{ color: #dc2626; }}
  .chart-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(450px, 1fr)); gap: 20px; margin-bottom: 24px; }}
  .chart-card {{ background: white; border-radius: 12px; padding: 20px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); }}
  .chart-card h3 {{ font-size: 16px; font-weight: 600; margin-bottom: 16px; color: #334155; }}
  .chart-card canvas {{ max-height: 300px; }}
  .table-card {{ background: white; border-radius: 12px; padding: 20px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); margin-bottom: 24px; }}
  .table-card h3 {{ font-size: 16px; font-weight: 600; margin-bottom: 16px; color: #334155; }}
  table {{ width: 100%; border-collapse: collapse; font-size: 14px; }}
  th {{ text-align: left; padding: 10px 12px; background: #f8fafc; border-bottom: 2px solid #e2e8f0; font-weight: 600; color: #475569; }}
  td {{ padding: 10px 12px; border-bottom: 1px solid #f1f5f9; }}
  .badge {{ display: inline-block; padding: 2px 10px; border-radius: 12px; font-size: 12px; font-weight: 600; }}
  .badge.premium {{ background: #dcfce7; color: #166534; }}
  .badge.standard {{ background: #dbeafe; color: #1e40af; }}
  .badge.below {{ background: #fef3c7; color: #92400e; }}
  .badge.reject {{ background: #fee2e2; color: #991b1b; }}
  .footer {{ text-align: center; padding: 20px; color: #94a3b8; font-size: 12px; }}
</style>
</head>
<body>
<div class="header">
  <h1>🏭 Manufacturing Quality Control Dashboard</h1>
  <p>Generated {datetime.now().strftime("%B %d, %Y at %I:%M %p")} • Data: 90 days • {total_lines} Production Lines</p>
</div>
<div class="container">

  <!-- KPI Cards -->
  <div class="kpi-grid">
    <div class="kpi-card blue">
      <div class="label">Overall OEE</div>
      <div class="value">{avg_oee}%</div>
      <div class="sub">Across all lines</div>
    </div>
    <div class="kpi-card green">
      <div class="label">Average Yield</div>
      <div class="value">{avg_yield}%</div>
      <div class="sub">Good units / total</div>
    </div>
    <div class="kpi-card amber">
      <div class="label">Total Production</div>
      <div class="value">{total_units:,}</div>
      <div class="sub">Units produced</div>
    </div>
    <div class="kpi-card red">
      <div class="label">Total Defects</div>
      <div class="value">{total_defects:,}</div>
      <div class="sub">Across all products</div>
    </div>
    <div class="kpi-card green">
      <div class="label">Best Line</div>
      <div class="value">{best_line["production_line"]}</div>
      <div class="sub">OEE: {round(best_line["overall_oee"], 1)}%</div>
    </div>
    <div class="kpi-card red">
      <div class="label">Needs Attention</div>
      <div class="value">{worst_line["production_line"]}</div>
      <div class="sub">OEE: {round(worst_line["overall_oee"], 1)}%</div>
    </div>
  </div>

  <!-- Charts Row 1 -->
  <div class="chart-grid">
    <div class="chart-card">
      <h3>📈 Daily OEE Trend by Production Line</h3>
      <canvas id="oeeTrend"></canvas>
    </div>
    <div class="chart-card">
      <h3>📊 OEE Classification Distribution</h3>
      <canvas id="oeeClass"></canvas>
    </div>
  </div>

  <!-- Charts Row 2 -->
  <div class="chart-grid">
    <div class="chart-card">
      <h3>🔧 Equipment Health Status</h3>
      <canvas id="healthChart"></canvas>
    </div>
    <div class="chart-card">
      <h3>🔄 Shift Performance Comparison</h3>
      <canvas id="shiftChart"></canvas>
    </div>
  </div>

  <!-- Charts Row 3 -->
  <div class="chart-grid">
    <div class="chart-card">
      <h3>📅 Weekly OEE & Production Trend</h3>
      <canvas id="weeklyTrend"></canvas>
    </div>
    <div class="chart-card">
      <h3>🏭 Line Scorecard — OEE Comparison</h3>
      <canvas id="lineOEE"></canvas>
    </div>
  </div>

  <!-- Product Quality Table -->
  <div class="table-card">
    <h3>🏆 Product Quality Rankings</h3>
    <table>
      <thead>
        <tr><th>#</th><th>Product</th><th>Avg Yield %</th><th>Total Units</th><th>Quality Tier</th></tr>
      </thead>
      <tbody>
''' + ''.join([f'''        <tr>
          <td>{p["quality_rank"]}</td>
          <td><strong>{p["product"]}</strong></td>
          <td>{round(p["avg_yield"], 1)}%</td>
          <td>{int(p["total_units"]):,}</td>
          <td><span class="badge {p["quality_tier"].lower().replace(" ", "").replace("belowtarget","below").replace("rejectreview","reject")}">{p["quality_tier"]}</span></td>
        </tr>\n''' for p in product_data]) + f'''      </tbody>
    </table>
  </div>

  <!-- Line Scorecard Table -->
  <div class="table-card">
    <h3>📋 Production Line Scorecard</h3>
    <table>
      <thead>
        <tr><th>Line</th><th>OEE %</th><th>Yield %</th><th>Total Units</th><th>Defects</th><th>Operating Days</th><th>Daily Output</th></tr>
      </thead>
      <tbody>
''' + ''.join([f'''        <tr>
          <td><strong>{row["production_line"]}</strong></td>
          <td>{round(row["overall_oee"], 1)}%</td>
          <td>{round(row["overall_yield"], 1)}%</td>
          <td>{int(row["total_units_all_time"]):,}</td>
          <td>{int(row["total_defects_all_time"]):,}</td>
          <td>{int(row["operating_days"])}</td>
          <td>{int(row["units_per_day"]):,}</td>
        </tr>\n''' for _, row in df_scorecard.sort_values('overall_oee', ascending=False).iterrows()]) + '''      </tbody>
    </table>
  </div>

</div>
<div class="footer">Fabric Demo Gallery — Manufacturing Quality Control Analytics</div>

<script>
const lineColors = ["#2563eb", "#16a34a", "#d97706", "#dc2626", "#8b5cf6", "#06b6d4"];

// OEE Trend
new Chart(document.getElementById("oeeTrend"), {
  type: "line",
  data: {
    labels: ''' + json.dumps(list(oee_lines.values())[0]['dates'] if oee_lines else []) + ''',
    datasets: ''' + json.dumps([{'label': line, 'data': data['values'], 'borderColor': ['#2563eb','#16a34a','#d97706','#dc2626'][i % 4], 'tension': 0.3, 'pointRadius': 1, 'borderWidth': 2, 'fill': False} for i, (line, data) in enumerate(oee_lines.items())]) + '''
  },
  options: { responsive: true, plugins: { legend: { position: "bottom" } }, scales: { y: { title: { display: true, text: "OEE %" } } } }
});

// OEE Class
new Chart(document.getElementById("oeeClass"), {
  type: "doughnut",
  data: {
    labels: ''' + json.dumps(list(oee_class_dist.keys())) + ''',
    datasets: [{ data: ''' + json.dumps(list(oee_class_dist.values())) + ''', backgroundColor: ''' + json.dumps([oee_class_colors.get(k, '#94a3b8') for k in oee_class_dist.keys()]) + ''' }]
  },
  options: { responsive: true, plugins: { legend: { position: "bottom" } } }
});

// Equipment Health
new Chart(document.getElementById("healthChart"), {
  type: "doughnut",
  data: {
    labels: ''' + json.dumps(list(health_dist.keys())) + ''',
    datasets: [{ data: ''' + json.dumps(list(health_dist.values())) + ''', backgroundColor: ''' + json.dumps([health_colors.get(k, '#94a3b8') for k in health_dist.keys()]) + ''' }]
  },
  options: { responsive: true, plugins: { legend: { position: "bottom" } } }
});

// Shift Comparison
new Chart(document.getElementById("shiftChart"), {
  type: "bar",
  data: {
    labels: ''' + json.dumps(shift_data['shift'].tolist()) + ''',
    datasets: [
      { label: "Units Produced", data: ''' + json.dumps([int(v) for v in shift_data['total_units'].tolist()]) + ''', backgroundColor: "#3b82f6" },
      { label: "Defects", data: ''' + json.dumps([int(v) for v in shift_data['total_defects'].tolist()]) + ''', backgroundColor: "#ef4444" }
    ]
  },
  options: { responsive: true, plugins: { legend: { position: "bottom" } } }
});

// Weekly Trend
new Chart(document.getElementById("weeklyTrend"), {
  type: "bar",
  data: {
    labels: ''' + json.dumps(['Week ' + str(int(w)) for w in weekly_agg['production_week'].tolist()]) + ''',
    datasets: [
      { label: "Weekly Units", data: ''' + json.dumps([int(v) for v in weekly_agg['weekly_units'].tolist()]) + ''', backgroundColor: "#3b82f680", borderColor: "#3b82f6", borderWidth: 1, yAxisID: "y" },
      { label: "Avg OEE %", data: ''' + json.dumps([round(v, 1) for v in weekly_agg['weekly_avg_oee'].tolist()]) + ''', type: "line", borderColor: "#16a34a", tension: 0.3, pointRadius: 4, yAxisID: "y1" }
    ]
  },
  options: { responsive: true, plugins: { legend: { position: "bottom" } }, scales: { y: { title: { display: true, text: "Units" } }, y1: { position: "right", title: { display: true, text: "OEE %" }, grid: { drawOnChartArea: false } } } }
});

// Line OEE
new Chart(document.getElementById("lineOEE"), {
  type: "bar",
  data: {
    labels: ''' + json.dumps(df_scorecard.sort_values('overall_oee', ascending=False)['production_line'].tolist()) + ''',
    datasets: [{ label: "Overall OEE %", data: ''' + json.dumps([round(v, 1) for v in df_scorecard.sort_values('overall_oee', ascending=False)['overall_oee'].tolist()]) + ''', backgroundColor: lineColors }]
  },
  options: { responsive: true, indexAxis: "y", plugins: { legend: { display: false } }, scales: { x: { title: { display: true, text: "OEE %" }, max: 100 } } }
});
</script>
</body>
</html>'''

print(f'Dashboard HTML generated: {len(html):,} bytes')

In [ ]:
# Save dashboard to lakehouse Files
with open('/lakehouse/default/Files/dashboard.html', 'w', encoding='utf-8') as f:
    f.write(html)
print('Dashboard saved to Files/dashboard.html')
print('\nTo view: Open the lakehouse > Files > dashboard.html > Download and open in browser')

In [ ]:
# Display inline preview
displayHTML(html)